# Quick Start 3 - Portfolio Builder

Compare ranking, linear and mean-variance portfolio builders.


In [ ]:
import asyncio
import datetime as dt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.portfolio.linearbuilder import linear_builder
from alphamind.portfolio.meanvariancebuilder import mean_variance_builder
from alphamind.portfolio.rankbuilder import rank_build

latest = panel.filter(pc.equal(panel["trade_date"], dates[-1]))
signal = latest["factor"].to_numpy(zero_copy_only=False)
ranked = rank_build(signal, use_rank=3).flatten().astype(float) / 3.0
risk = np.ones((n_assets, 1))
target = (np.array([1.0]), np.array([1.0]))
linear_status, _, linear_weights = linear_builder(signal, 0.0, 1.0, risk, target)
benchmark = np.full(n_assets, 1.0 / n_assets)
risk_model = {"cov": np.eye(n_assets) * 0.02, "factor_cov": None, "factor_loading": None, "idsync": None}
mv_status, _, mv_weights = mean_variance_builder(
    signal, risk_model, benchmark, 0.0, 1.0, risk, target
)
assert ranked.sum() == 1.0
assert linear_status in ("optimal", "optimal_inaccurate")
assert mv_status in ("optimal", "optimal_inaccurate")
pa.table({"code": latest["code"], "ranked": ranked, "linear": linear_weights, "mv": mv_weights})
